# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`). Each entity (record set, field, column) should be referenced using its Croissant `@id`.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets and Fields:")
record_set_ids = []
for record_set in metadata.record_sets:
    print(f"- Record Set @id: {record_set.id} | Name: {record_set.name}")
    record_set_ids.append(record_set.id)
    for field in record_set.fields:
        print(f"   - Field @id: {field.id} | Name: {field.name} | DataType: {getattr(field, 'data_type', 'N/A')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Below we load all record sets and preview the columns in the first table.

In [ ]:
# Prepare to extract all record sets
dataframes = dict()

# Let's print all record set ids (as above)
print(f"Record Set IDs: {record_set_ids}\n")

# Example: load all record sets to DataFrames using their unique @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")

# Let's assign the main record set id to use below
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping. We will work with a numeric field from the main record set identified above.

In [ ]:
import numpy as np

df = dataframes.get(main_record_set_id)

# Print all columns so user can choose a numeric field
print("Available columns in main record set:", df.columns.tolist())

# Attempt to auto-select a numeric field commonly found in mass spectrometry datasets
possible_numeric_fields = ['MW', 'Molecular Weight', 'Coverage', 'Unique Peptides', 'Total Peptides', 'pI']
numeric_field = None
for col in possible_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break

# If auto-selection fails, fall back to the first numeric column
if numeric_field is None:
    # Try to infer by dtype
    for col in df.columns:
        # Heuristics for numeric data
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break

if not numeric_field:
    raise ValueError("No numeric field found. Please inspect columns and select an appropriate numeric field.")

print(f"Using numeric field for EDA: {numeric_field}")

# Remove records with missing/invalid values in this field
filtered_df = df[df[numeric_field].apply(lambda x: isinstance(x, (int, float, np.integer, np.floating)))]
# Or attempt to coerce values to numeric
filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
filtered_df = filtered_df.dropna(subset=[numeric_field])

threshold = filtered_df[numeric_field].quantile(0.75)  # Use 75th percentile as example
filtered_high = filtered_df[filtered_df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):")
display(filtered_high.head())

# Normalize
filtered_high[f"{numeric_field}_normalized"] = (filtered_high[numeric_field] - filtered_high[numeric_field].mean()) / filtered_high[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_high[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to find a grouping field (e.g., a categorical/label field)
possible_group_fields = ['Sample', 'Condition', 'Type']
group_field = None
for col in possible_group_fields:
    if col in df.columns:
        group_field = col
        break

if group_field:
    grouped_df = filtered_high.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())
else:
    print("No obvious categorical/group field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field exists, show boxplot
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_high)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to programmatically access and explore the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from human mast cells.

- We listed all available record sets and referenced all entities by their Croissant `@id`.
- Data from the main record set was loaded and analyzed; common EDA steps such as filtering, normalization, and summary statistics were demonstrated.
- Visualizations provided further insight into the distribution and group differences in key numeric fields.

For deeper analysis, you may continue to explore additional record sets, fields, or join tables depending on your research question, always referencing by `@id` as per Croissant best practice.